# 01 — Amazon Reviews Preprocessing

**입력**: `data/bronze/amazon/{brand}_items.csv`, `{brand}_reviews.csv` (5 브랜드) + `skinsort_0115.csv`

**출력**:
- `data/silver/amazon/amazon_reviews_lemmatized.csv` — 5 브랜드 통합, lemmatize + n-gram 전처리 완료
- `data/silver/amazon/amazon_items_processed.csv` — 아이템 정보 (가격·카테고리·평점 등)
- `data/silver/amazon/skinsort_processed.csv` — Skinsort Korean 브랜드 데이터 정제본

**범위**: 결측치 처리 → 컬럼 변환 → 텍스트 전처리 (lemmatize + bigram/trigram)

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
                 if (p / '.git').is_dir())
sys.path.insert(0, str(REPO_ROOT / 'src'))
from util.repo_paths import BRONZE_AMAZON, SILVER_AMAZON

# amazon_nlp util — NLP 함수 (clean_text 등) 통합 (M4 리팩터).
# 자세히: src/util/amazon_nlp.py
from util.amazon_nlp import (
    clean_text, tokenize_text, remove_stopwords,
    stem_tokens, lemmatize_tokens,
    bigram_filter, trigram_filter,
)


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast
import datetime
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import StandardScaler
from yellowbrick.cluster import KElbowVisualizer
from yellowbrick.cluster import intercluster_distance
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 데이터셋 불러오기
dr_items = pd.read_csv(BRONZE_AMAZON / 'Dr_jart_items.csv')
dr_reviews = pd.read_csv(BRONZE_AMAZON / 'Dr_jart_reviews.csv')
cs_items = pd.read_csv(BRONZE_AMAZON / 'cosrx_items.csv')
cs_reviews = pd.read_csv(BRONZE_AMAZON / 'cosrx_reviews.csv')
if_items = pd.read_csv(BRONZE_AMAZON / 'imfrom_items.csv')
if_reviews = pd.read_csv(BRONZE_AMAZON / 'imfrom_reviews.csv')
bj_items = pd.read_csv(BRONZE_AMAZON / 'joseon_items.csv')
bj_reviews = pd.read_csv(BRONZE_AMAZON / 'joseon_reviews.csv')
pu_items = pd.read_csv(BRONZE_AMAZON / 'purito_items.csv')
pu_reviews = pd.read_csv(BRONZE_AMAZON / 'purito_reviews.csv')
skinsort = pd.read_csv(BRONZE_AMAZON / 'skinsort_0115.csv')

## Data Overview

In [ ]:
# 전체적인 데이터 정보 EDA
def eda_overveiw(df):

    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)

    print(f"\n=================== DATA OVERVIEW ===================")

    # 데이터 상단부 확인
    print("\n----------------- Head -----------------")
    print(df.head())
    print("=" * 60)

    # 데이터 정보 및 크기 확인
    print("\n--------------- Information ---------------")
    print(df.info())
    print(f"\nSize: {df.size}")
    print(f"Shape: {df.shape}")
    print("=" * 60)

    # 결측치 확인
    print("\n--------------- Missing Values ---------------")
    print(df.isnull().sum())
    print("=" * 60)

    # 중복값 확인
    print("\n--------------- Duplicate Values ---------------")
    print(df.duplicated().value_counts())
    print("=" * 60)

    # 데이터 기술통계량 확인 (int, float type)
    print("\n------------ Descriptive Statistics (Numeric) ------------")
    print(df.describe())
    print("=" * 60)

    # 데이터 기술통계량 확인 (object type)
    print("\n------------ Descriptive Statistics (Categorical) ------------")

    try:
        print(df.describe(include=[object]))

    except:
        object_cols = df.select_dtypes(include=['object']).columns
        error_cols = []

        for col in object_cols:
            if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
                error_cols.append(col)
        print(f"{error_cols} Excluded")
        print(df.loc[:, ~df.columns.isin(error_cols)].describe(include=[object]))

    print("=" * 60)

eda_overveiw(dr_items)

## Data Cleaning

### Copied Dataset

In [ ]:
# 분석 & 전처리용으로 원본 데이터 복사
dr_items_copy = dr_items.copy()
dr_reviews_copy = dr_reviews.copy()
cs_items_copy = cs_items.copy()
cs_reviews_copy = cs_reviews.copy()
if_items_copy = if_items.copy()
if_reviews_copy = if_reviews.copy()
bj_items_copy = bj_items.copy()
bj_reviews_copy = bj_reviews.copy()
pu_items_copy = pu_items.copy()
pu_reviews_copy = pu_reviews.copy()
skinsort_copy = skinsort.copy()

### Handling Missing Values

In [ ]:
# 'brand'가 없거나 다른 데이터 확인 -> 각 파일마다 다르게 정리 필요
# 이유 - 브랜드별 아이템만 추출했기 때문 + 세부사항에 브랜드명 작성되어 있으니 그것보고 판단

# 브랜드 명 변경
brand_list = ['Dr.Jart+', 'COSRX', "I'm from", 'Beauty of Joseon','PURITO']
dr_items_copy['brand'].replace('No brand', brand_list[0], inplace=True)
cs_items_copy['brand'].replace('No brand', brand_list[1], inplace=True)
if_items_copy['brand'].replace('No brand', brand_list[2], inplace=True)
bj_items_copy['brand'].replace('No brand', brand_list[3], inplace=True)
pu_items_copy['brand'].replace('No brand', brand_list[4], inplace=True)

# items_df들의 컬럼 수정을 위해 df를 list에 포함시키기
items_list = [dr_items_copy, cs_items_copy, if_items_copy, bj_items_copy, pu_items_copy]

# reviews_df들의 컬럼 수정을 위해 df를 list에 포함시키기
reviews_list = [dr_reviews_copy, cs_reviews_copy, if_reviews_copy, bj_reviews_copy, pu_reviews_copy]

In [ ]:
# items_df들의 결측치를 NaN 값으로 한번에 처리
def preprocess_items(df_items):

    df_items['best_sellers_rank_Feature'].replace('No result', np.nan, inplace=True)
    df_items['global_rating_count'].replace('No rating', np.nan, inplace=True)
    df_items['Special_Feature'].replace('No special feature', np.nan, inplace=True)

for i in range(len(items_list)):
    preprocess_items(items_list[i])

# reviews_df들의 결측치를 NaN 값으로 한번에 처리
def preprocess_reviews(df_reviews):

    df_reviews['date'].replace('No date', np.nan, inplace=True)
    df_reviews['review_rating'].replace('No review', np.nan, inplace=True)

for i in range(len(reviews_list)):
    preprocess_reviews(reviews_list[i])

# review_df : reivew content 결측치 제거 함수
def dropna_reviews(reviews_df):
    reviews_df.dropna(inplace=True)
    reviews_df.reset_index(inplace=True)

# review data 결측치 제거
for i in range(len(reviews_list)):
    dropna_reviews(reviews_list[i])

# 카테고리별 EDA를 진행하기 위한 아이템 데이터셋 생성
amazon_items_df = pd.concat([dr_items_copy, cs_items_copy, if_items_copy, bj_items_copy, pu_items_copy])

In [ ]:
# skinsort 결측치 처리 및 확인
skinsort_copy.dropna(subset=['country','afterUse', 'type'], inplace=True)
skinsort_copy.reset_index(inplace=True)

# 결측치 삭제 확인 시각화
plt.close()
plt.figure(figsize=(9,8))
sns.heatmap(skinsort_copy.isnull(), cbar_kws={'label':'Missing Values'}, cmap='Oranges')
plt.title('Drop Missing Values in Profile Dataset')
plt.show()

### Column-wise data transformation

In [ ]:
# df_items : description 컬럼
def preprocess_description(df_items):

    for i in range(len(df_items)):
        description = ast.literal_eval(df_items.description[i])

        for key, value in description.items():
            if key not in df_items.columns:
                df_items[key] = np.nan
            df_items.loc[i, key] = value

    df_items.drop(columns=['description'], inplace=True)

# df_items : detail_dict 컬럼
def preprocess_detail_dict(df_items):

    for i in range(len(df_items)):
        detail_dict = ast.literal_eval(df_items.detail_dict[i])

        for key, value in detail_dict.items():
            if key not in df_items.columns:
                df_items[key] = np.nan
            df_items.loc[i, key] = value

    df_items.drop(columns=['detail_dict'], inplace=True)

# df_items : best_sellers_rank_Feature 컬럼을 세부 컬럼으로 나누기
def preprocess_best_sellers_col(df_items):

        for i in range(len(df_items)):
            try:
                value = df_items.best_sellers_rank_Feature[i]

                if pd.isna(value): #np.nan 인 애들
                    # print(f"{i}, {df_items.best_sellers_rank_Feature[i]} Passed")
                    continue

                if isinstance(value, float): # float 타입인 애들
                    value = str(value)
                    print(f"{i}, {df_items.best_sellers_rank_Feature[i]} Passed")

                cat_list = df_items.best_sellers_rank_Feature[i].split('#')

                df_items.loc[i, 'Category'] = cat_list[1]
                df_items.loc[i, 'Sub_Category'] = cat_list[2]

                detail_list = df_items.Category[i].split('in')

                df_items.loc[i, 'Category_Rank']= detail_list[0]
                df_items.loc[i, 'Category_Name']= detail_list[1]

                df_items.loc[i, 'Category_Name'] = df_items.Category_Name[i].split('(')[0]

                sub_list = df_items.Sub_Category[i].split('in')

                df_items.loc[i, 'Sub_Category_Rank']= sub_list[0]
                df_items.loc[i, 'Sub_Category_Name']= sub_list[1]

            except Exception as e:
                print(f"{i}, {df_items.best_sellers_rank_Feature[i]} Passed")
                print(f"Error: {e}")

                continue

        df_items.drop(columns=['best_sellers_rank_Feature'], inplace=True)

# df_items들의 칼럼 수정 한번에 진행
for i in range(len(items_list)) :
    preprocess_description(items_list[i])
    preprocess_detail_dict(items_list[i])
    preprocess_best_sellers_col(items_list[i])

In [ ]:
# df_reviews : date, rating 컬럼
def preprocess_review_cols(df_reviews):

    for i in range(len(df_reviews)):

        try:

            if pd.isna(df_reviews.date[i]) or 'on' not in df_reviews.date[i]:
                # print(f"{i}, {df_reviews.date[i]} Passed")
                continue
            else:
                df_reviews.loc[i, 'review_date'] = df_reviews.date[i].split('on')[1]
                df_reviews.loc[i, 'review_date'] = pd.to_datetime(df_reviews.review_date[i])

            if pd.isna(df_reviews.review_rating[i]):
                print(f"{i}, {df_reviews.review_rating[i]} Passed")
                continue
            else:
                df_reviews.loc[i, 'review_rating'] = float(df_reviews.review_rating[i].split('out')[0])
        except Exception as e:
            print(f"{i}, {df_reviews.review_rating[i]}, Error: {e}")
            continue

# df_items들의 컬럼 한번에 수정하기: 'review_date'를 날짜 형식으로 변경, review_rating을 정수형으로 변경
for i in range(len(reviews_list)) :
    preprocess_review_cols(reviews_list[i])
    reviews_list[i]['review_date'] = pd.to_datetime(reviews_list[i]['review_date'])
    reviews_list[i].drop(columns=['date'], inplace=True)
    reviews_list[i]['review_rating'] = pd.to_numeric(reviews_list[i]['review_rating'], errors='coerce', downcast='integer')

In [ ]:
# 브랜드별 item, review 데이터 merge 결합
dr_df = pd.merge(dr_items_copy, dr_reviews_copy, on='ASIN')
cs_df = pd.merge(cs_items_copy, cs_reviews_copy, on='ASIN')
if_df = pd.merge(if_items_copy, if_reviews_copy, on='ASIN')
bj_df = pd.merge(bj_items_copy, bj_reviews_copy, on='ASIN')
pu_df = pd.merge(pu_items_copy, pu_reviews_copy, on='ASIN')

merge_list = [dr_df, cs_df, if_df, bj_df, pu_df]

In [ ]:
def preprocess_merge_df(merge_df):

    # 컬럼명 변경, 불필요 컬럼 삭제, 인덱스 수정, 결측치 확인
    merge_df.rename(columns={'content':'review_content', 'title_x':'title'}, inplace=True)
    merge_df.drop(['title_y'], axis=1, inplace=True)
    merge_df.reset_index(drop=True, inplace=True)
    merge_df.isnull().sum()

    # Category_Rank
    merge_df['Category_Rank'] = merge_df['Category_Rank'].apply(
        lambda x: str(x).replace(',', '') if not pd.isna(x) and isinstance(x, float) else x
    ).astype('str').apply(lambda x: x.replace(',', '') if x != 'nan' else None)

    # Sub_Category_Rank
    merge_df['Sub_Category_Rank'] = merge_df['Sub_Category_Rank'].apply(
        lambda x: str(x).replace(',', '') if not pd.isna(x) and isinstance(x, float) else x
    ).astype('str').apply(lambda x: x.replace(',', '') if x != 'nan' else None)

    # review_rating
    merge_df['review_rating'] = merge_df['review_rating'].apply(
        lambda x: str(x) if not pd.isna(x) and isinstance(x, float) else x
    ).astype('float', errors='ignore')

    # review_date
    merge_df['review_date'] = merge_df['review_date'].apply(
        lambda x: pd.to_datetime(x) if not pd.isna(x) else None
    )

    merge_df.reset_index(drop=True, inplace=True)

    merge_df.info()
    merge_df.head(2)
    # amazon_df.to_csv(DATA_PATH + 'amazon_df_0116.csv', encoding='utf-8')

In [ ]:
# 언어 감지 및 번역을 위한 라이브러리: langdetect 간편하게 사용할 수 있어서 선택, Fasttext는 사전 학습된 모델을 다운 받아야 해서 미사용

from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from deep_translator import GoogleTranslator

# 언어 감지
def detect_language(text):

    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

# 영어가 아닌 언어를 구글번역을 활용하여 영어로 번역
def translate_en(text):

    to_translate = text
    translated = GoogleTranslator(source='auto', target='english').translate(to_translate)

    return translated

In [ ]:
def text_preprocessing(merge_df) :

    # str 형식이 아닐 경우 오류 방지를 위한 코드
    merge_df['review_content'] = merge_df['review_content'].fillna('').astype(str)

    # 리뷰 언어가 영어가 아닌 경우 -> 번역
    #merge_df['detected_language'] = merge_df['review_content'].apply(detect_language)
    merge_df['detected_language'] = merge_df['review_content'].apply(lambda x: detect_language(x) if isinstance(x, str) else None)

    merge_df.loc[merge_df['detected_language'] != 'en', 'review_content'] = merge_df.loc[merge_df['detected_language'] != 'en', 'review_content'].apply(translate_en)

    # merge_df.to_csv(DATA_PATH + AMAZON / 'amazon_koreaOnly_translated.csv', encoding='utf-8')
    merge_df.rename(columns={'category':'Amazon_Category'}, inplace=True)
    merge_df['review_date'] = merge_df['review_date'].apply(lambda x: pd.to_datetime(x) if not pd.isna(x) else None)
    merge_df['global_rating_count'] = merge_df['global_rating_count'].astype('Int64')
    merge_df['Category_Rank'] = merge_df['Category_Rank'].astype('Int64')
    merge_df['Sub_Category_Rank'] = merge_df['Sub_Category_Rank'].astype('Int64')

    print(merge_df.head(2))

# 브랜드 별 아이템, 리뷰 데이터 merge로 결합 후 리뷰데이터 번역 진행
# 시간이 다소 소요
for i in range(len(merge_list)):
    preprocess_merge_df(merge_list[i])
    text_preprocessing(merge_list[i])

In [ ]:
# 스킨쏘트 - 브랜드별 성분, 사용후효과 컬럼럼
def preprocess_skinsort_col(text):
    spaced_text = re.sub(r',(?!\s)', ', ', text)  # , 뒤에 공백이 없으면 추가
    return spaced_text

skinsort_copy['ingridients'] = skinsort_copy['ingridients'].apply(preprocess_skinsort_col)
skinsort_copy['afterUse'] = skinsort_copy['afterUse'].apply(preprocess_skinsort_col)

### Base dataset

In [ ]:
# 각 브랜드별 결측치 및 전처리 실시 후 결합
amazon_df = pd.concat([dr_df, cs_df, if_df, bj_df, pu_df])
############# 삭제할 컬럼 다같이 결정 필요 #############

### Text preprocessing

In [ ]:
# nltk 에서 Punkt tokenizer & stopwords list를 다운로드
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from nltk.util import ngrams
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

# # lda 모델링 추가 불용어
# lda_stopwords = [
#     "would", "use", "using","locals", "cf", "fb", "ba" , "use", "review", "brand", "brands", "item", "items", "category", "categories", "line", "lines","formula", "formulas", "collection", "collections",
# "really","skin","used","time", "makes","tried","one","skin feel","lot","trying","buy","apply","quite","way","never","bought", "cosrx","always","less",
# "time","without","absolutely","might","maybe","sure","think","though", "getting","result","know", "especially","dr jart","feel","purchase","definitely","im","thing","started","need","another","noticed","actually", "people","got","box","every","another","found","jart","wear"
# "set", "sets", "value", "values", "pack", "packs","new","ive"]

# # K-beauty 추가 불용어
# kbeauty_stopwords =  {
# "and", "beauty", "skincare", "cosmetics", "product", "products","use", "using", "from","for","im","i'm","floz",
# "skin", "care", "makeup", "mask", "sheet","best", "top", "favorite",
# "amazing", "perfect", "good", "bad", "recommend", "use", "review", "love","brand",
# "brands", "item", "items", "category", "categories", "line", "lines","formula", "formulas", "ingredient", "ingredients",
# "collection", "collections","set", "sets", "value", "values", "pack", "packs","latest", "exclusive", "limited",
# "special", "popular", "quality", "For",
# "safe", "worked", "works", " product"
# "face", "feel","really","stuff","joseon",
# "skin","used","time", "dont","makes","tried","one","skin feel","lot","trying","buy","apply","quite","way","never",
# "bought", "always","without","absolutely","might","maybe","sure","think","though",
# "getting","want","result","know", "especially","dr jart","purchase","definitely",
# "thing","started","need","type","facial","another","noticed","actually",
# "people","money","got","box","every","another","found","jart","wear","drjart","nan","1","no","non","not", "drjart","to",
# "cosrx","From" , "drjrt", "types", "of", " of", "100ml","200ml","300ml","150ml","250ml","50ml","30ml", "338","507","676","purito"
# }

stop_words = stopwords.words('english')
# stop_words.extend(['from', 'subject', 're', 'edu', 'use','object','generator','genexpr'])
# stop_words.extend(lda_stopwords)
# stop_words.extend(kbeauty_stopwords)

# clean_text 는 util.amazon_nlp 에서 import.
    else:
        return text

# tokenize_text 는 util.amazon_nlp 에서 import.
# remove_stopwords 는 util.amazon_nlp 에서 import (stop_words 인자 명시).
# stem_tokens 는 util.amazon_nlp 에서 import (stemmer 인자 명시).
# lemmatize_tokens 는 util.amazon_nlp 에서 import (lemmatizer 인자 명시).
# 저장
amazon_df['cleaned_title'] = amazon_df['title'].apply(clean_text)
amazon_df['cleaned_review'] = amazon_df['review_content'].apply(clean_text)
skinsort_copy['cleaned_ingridients'] = skinsort_copy['ingridients'].apply(clean_text)
skinsort_copy['cleaned_afterUse'] = skinsort_copy['afterUse'].apply(clean_text)

amazon_df['tokenized_title'] = amazon_df['cleaned_title'].apply(tokenize_text)
amazon_df['tokenized_review'] = amazon_df['cleaned_review'].apply(tokenize_text)
skinsort_copy['tokenized_ingridients'] = skinsort_copy['cleaned_ingridients'].apply(tokenize_text)
skinsort_copy['tokenized_afterUse'] = skinsort_copy['cleaned_afterUse'].apply(tokenize_text)

amazon_df['stpw_processed_title'] = amazon_df['tokenized_title'].apply(lambda t: remove_stopwords(t, stop_words))
amazon_df['stpw_processed_review'] = amazon_df['tokenized_review'].apply(lambda t: remove_stopwords(t, stop_words))
skinsort_copy['stpw_processed_ingridients'] = skinsort_copy['tokenized_ingridients'].apply(lambda t: remove_stopwords(t, stop_words))
skinsort_copy['stpw_processed_afterUse'] = skinsort_copy['tokenized_afterUse'].apply(lambda t: remove_stopwords(t, stop_words))

amazon_df['stemmed_title'] = amazon_df['stpw_processed_title'].apply(lambda t: stem_tokens(t, stemmer))
amazon_df['stemmed_review'] = amazon_df['stpw_processed_review'].apply(lambda t: stem_tokens(t, stemmer))
skinsort_copy['stemmed_ingridients'] = skinsort_copy['stpw_processed_ingridients'].apply(lambda t: stem_tokens(t, stemmer))
skinsort_copy['stemmed_afterUse'] = skinsort_copy['stpw_processed_afterUse'].apply(lambda t: stem_tokens(t, stemmer))

# bigram 하기 전
amazon_df['lemmatized_title'] = amazon_df['stpw_processed_title'].apply(lambda t: lemmatize_tokens(t, lemmatizer))
amazon_df['lemmatized_review'] = amazon_df['stpw_processed_review'].apply(lambda t: lemmatize_tokens(t, lemmatizer))
skinsort_copy['lemmatized_ingridients'] = skinsort_copy['stpw_processed_ingridients'].apply(lambda t: lemmatize_tokens(t, lemmatizer))
skinsort_copy['lemmatized_afterUse'] = skinsort_copy['stpw_processed_afterUse'].apply(lambda t: lemmatize_tokens(t, lemmatizer))

amazon_df.head(2)

https://towardsdatascience.com/6-tips-to-optimize-an-nlp-topic-model-for-interpretability-20742f3047e2

In [ ]:
from collections import Counter

word_counts = Counter(" ".join([" ".join(review) for review in amazon_df['lemmatized_review'].dropna()]).split())
common_words = word_counts.most_common(50)  # 가장 많이 등장하는 50개 단어 확인
print(common_words)

In [ ]:
amazon_df.cleaned_review[0]

In [ ]:
# bigram 
bigram_measures = nltk.collocations.BigramAssocMeasures()
finder = nltk.collocations.BigramCollocationFinder.from_documents([comment.split() for comment in amazon_df.cleaned_review if isinstance(comment, str)])

# # 불용어 제거 반영된 상태에서 진행
# finder.apply_word_filter(lambda w: w in stop_words)

# Filter only those that occur at least 40 times
finder.apply_freq_filter(40)
bigram_scores = finder.score_ngrams(bigram_measures.pmi)
print("len of bigram_scores: ", len(bigram_scores))
bigram_scores[:10]

In [ ]:
# trigram
trigram_measures = nltk.collocations.TrigramAssocMeasures()
finder = nltk.collocations.TrigramCollocationFinder.from_documents([comment.split() for comment in amazon_df.cleaned_review if isinstance(comment, str)])
# finder.apply_word_filter(lambda w: w in stop_words)
# Filter only those that occur at least 40 times
finder.apply_freq_filter(40)
trigram_scores = finder.score_ngrams(trigram_measures.pmi)
print("len of trigram_scores: ", len(trigram_scores))
trigram_scores[:10]

In [ ]:
bigram_pmi = pd.DataFrame(bigram_scores)
bigram_pmi.columns = ['bigram', 'pmi']
bigram_pmi.sort_values(by='pmi', axis = 0, ascending = False, inplace = True)

trigram_pmi = pd.DataFrame(trigram_scores)
trigram_pmi.columns = ['trigram', 'pmi']
trigram_pmi.sort_values(by='pmi', axis = 0, ascending = False, inplace = True)

In [ ]:
# bigram_filter 는 util.amazon_nlp 에서 import (stop_words 인자 명시).
# trigram_filter 는 util.amazon_nlp 에서 import (stop_words 인자 명시).

In [ ]:
# Can set pmi threshold to whatever makes sense - eyeball through and select threshold where n-grams stop making sense
# choose top 500 ngrams in this case ranked by PMI that have noun like structures
filtered_bigram = bigram_pmi[bigram_pmi.apply(lambda bigram:\
                                            bigram_filter(bigram['bigram'])\
                                            and bigram.pmi > 5, axis = 1)][:500]

filtered_trigram = trigram_pmi[trigram_pmi.apply(lambda trigram: \
                                                trigram_filter(trigram['trigram'])\
                                                and trigram.pmi > 5, axis = 1)][:500]

bigrams = [' '.join(x) for x in filtered_bigram.bigram.values if len(x[0]) > 2 or len(x[1]) > 2]
trigrams = [' '.join(x) for x in filtered_trigram.trigram.values if len(x[0]) > 2 or len(x[1]) > 2 and len(x[2]) > 2]

In [ ]:
bigrams[:10]

In [ ]:
trigrams[:10]

In [ ]:
# replace_ngram 은 util.amazon_nlp 에서 import (M4 리팩터).
reviews_w_ngrams = pd.DataFrame(amazon_df.lemmatized_review.copy())
reviews_w_ngrams.cleaned_review = reviews_w_ngrams.lemmatized_review.map(lambda x: replace_ngram(x, bigrams, trigrams) if isinstance(x, str) else x)

# tokenize reviews + remove stop words + remove names + remove words with less than 2 characters
reviews_w_ngrams = reviews_w_ngrams.lemmatized_review.map(lambda x: [word for word in x.split() if word not in stop_words and len(word) > 2] if isinstance(x, str) else x)
reviews_w_ngrams.dropna(axis=0, inplace=True)
reviews_w_ngrams.isnull().sum()

reviews_w_ngrams.head()
final_reviews = reviews_w_ngrams.copy()

In [ ]:
# silver bridge — 02/03 노트북의 입력
SILVER_AMAZON.mkdir(parents=True, exist_ok=True)
amazon_df.to_csv(SILVER_AMAZON / 'amazon_reviews_lemmatized.csv', index=False)
amazon_items_df.to_csv(SILVER_AMAZON / 'amazon_items_processed.csv', index=False)
skinsort_copy.to_csv(SILVER_AMAZON / 'skinsort_processed.csv', index=False)
print('saved to:', SILVER_AMAZON)